# HyenaDNA: Long-Range Genomic Sequence Modeling at Single Nucleotide Resolution
**ArXivist-generated reproduction notebook**
Paper: https://arxiv.org/abs/2306.15794
Generated: 2026-07-11

This notebook walks through the HyenaDNA reproduction: it checks the environment, installs the
project, explains the model, loads official pretrained weights, runs a small fine-tuning loop on a
tiny synthetic dataset (no downloads needed), and shows the paper's reported results for comparison.

Runnable end-to-end on a Colab GPU (or CPU) in well under 30 minutes.

## 1. Environment check

In [ ]:
# Check Python version, GPU availability, and key dependencies
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU - training will be slow")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Installation

In [ ]:
# Install the project in editable mode (run once). Notebook lives in notebooks/, repo root is '..'.
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])

## 3. Paper overview

**Problem.** DNA is extremely long-range: regulatory elements can sit hundreds of thousands of bases
from the genes they control. Attention-based genomic models scale quadratically and are usually
limited to short, k-mer-tokenized contexts.

**Key idea.** HyenaDNA replaces attention with the **Hyena operator** — data-controlled gating of
*implicit long convolutions* evaluated by FFT in `O(L log L)`. This lets it model up to **1M tokens**
at **single-nucleotide resolution** (~500x longer context), training up to 160x faster than a Transformer.

**How this repo maps to the paper:**
- `models/pretrained.py` -> loads the released hg38-pretrained backbone (Sec 3-4)
- `models/hyena_operator.py` -> from-scratch order-2 Hyena operator (Sec 3, reference only)
- `data/tokenizer.py` -> single-nucleotide char tokenizer (Sec 3)
- Downstream fine-tuning -> GenomicBenchmarks / Nucleotide Transformer (Sec 4)

## 4. Component walkthrough: the tokenizer

HyenaDNA tokenizes DNA at **single-nucleotide** resolution: each of A/C/G/T/N (plus special tokens)
is one token. No k-mers, no BPE.

In [ ]:
from src.hyenadna.data.tokenizer import CharTokenizer

tok = CharTokenizer(max_len=16)
ids = tok.encode("ACGTACGTNNNN")
print("tokenizer:", tok)
print("encoded ACGTACGTNNNN ->", ids)
print("vocab size:", tok.vocab_size, "| pad id:", tok.pad_id)

## 5. Component walkthrough: the Hyena operator (reference)

The order-2 Hyena operator gates two implicit long convolutions:

$$ y = x_2 \odot \mathrm{FFTConv}(h_1,\; x_1 \odot \mathrm{FFTConv}(h_0, x_0)) $$

where $x_0, x_1, x_2$ are short-conv projections of the input and $h_0, h_1$ are implicit filters
produced by an MLP over positions. **Note:** the filter internals here are inferred from the Hyena
reference (SIR confidence 0.55) and are for study; the reproduction uses pretrained weights.

In [ ]:
import torch
from src.hyenadna.models.hyena_operator import HyenaOperator, HyenaBlock

d_model = 128
op = HyenaOperator(d_model=d_model)
x = torch.randn(2, 32, d_model)   # [B=2, L=32, D=128]
try:
    y = op(x)
    print(f"Input shape:  {tuple(x.shape)}")
    print(f"Output shape: {tuple(y.shape)}")
    print(f"Expected:     (2, 32, 128)")
    assert y.shape == x.shape
    print("Hyena operator forward pass OK")
except Exception as e:
    print(f"[error] {e}")

## 6. Load the pretrained HyenaDNA classifier

Primary path: pull official weights from HuggingFace (`LongSafari/hyenadna-tiny-1k-seqlen`) and
attach a classification head. If the download is unavailable, the code falls back to the from-scratch
reference backbone and prints a warning (numbers won't match the paper then).

In [ ]:
from src.hyenadna.models.pretrained import HyenaDNAClassifier

model = HyenaDNAClassifier.from_pretrained(
    variant="hyenadna-tiny-1k-seqlen",
    num_classes=2,
    d_model=128,
    pool="mean",
    device=str(device),
)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameter count: {n_params/1e6:.2f}M")

## Mini-training on synthetic data

No downloads: we build a tiny synthetic binary task where class 1 sequences are GC-rich and class 0
are AT-rich. A well-behaved model should drive the loss down within a few steps.

In [ ]:
import random, torch
from torch.utils.data import DataLoader
from src.hyenadna.data.dataset import GenomicDataset

random.seed(42)
def make_seq(gc_rich):
    pool = "GCGCGCAT" if gc_rich else "ATATATGC"
    return "".join(random.choice(pool) for _ in range(64))

seqs   = [make_seq(i % 2 == 0) for i in range(100)]
labels = [0 if i % 2 == 0 else 1 for i in range(100)]

ds = GenomicDataset(seqs, labels, CharTokenizer(max_len=64), max_len=64)
loader = DataLoader(ds, batch_size=16, shuffle=True)
print("synthetic dataset:", ds)

In [ ]:
# Reduced-config mini training loop (inline, not the full trainer)
import torch.nn.functional as F
mini = HyenaDNAClassifier.from_pretrained("hyenadna-tiny-1k-seqlen", num_classes=2,
                                          d_model=128, pool="mean", device=str(device))
opt = torch.optim.AdamW(mini.parameters(), lr=1e-3)
mini.train()
for step, (x, y) in enumerate(loader):
    x, y = x.to(device), y.to(device)
    opt.zero_grad()
    loss = F.cross_entropy(mini(x), y)
    loss.backward(); opt.step()
    print(f"step {step+1:2d} | loss {loss.item():.4f}")
    if step >= 7:
        break
print("mini-training complete (loss should trend down)")

In [ ]:
# Sanity: output shapes and a prediction
mini.eval()
with torch.no_grad():
    xb, yb = next(iter(loader))
    logits = mini(xb.to(device))
print("logits shape:", tuple(logits.shape), "(expected (16, 2))")
print("sample preds:", logits.argmax(-1)[:8].cpu().tolist())

## 7. Paper results comparison

In [ ]:
# Results reported in the paper (from SIR evaluation_protocol.reported_results)
paper_results = {
    "GenomicBenchmarks/human_nontata_promoters (acc %)": 96.6,
    "GenomicBenchmarks/human_vs_worm (acc %)":           96.6,
    "GenomicBenchmarks/human_regulatory (acc %)":        93.8,
    "GenomicBenchmarks/coding_vs_intergenomic (acc %)":  91.3,
    "NucleotideTransformer/enhancer (MCC)":              62.6,
    "NucleotideTransformer/H3K4me3 (MCC)":               61.2,
}
print("Paper's reported results:")
for k, v in paper_results.items():
    print(f"  {k:52s} {v}")
print("\nTo reproduce: download a dataset, then run train.py with the full config.")
print("  python data/download.py genomic-benchmarks --dataset human_nontata_promoters")
print("  python train.py --config configs/config.yaml")
print("Then feed your numbers to ArXivist's Results Comparator (Stage 6).")

## What to do next

1. **Download data (API):** `python data/download.py genomic-benchmarks --dataset human_nontata_promoters`
2. **Full fine-tune:** `python train.py --config configs/config.yaml`
3. **Evaluate:** `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/best.pt`
4. **Compare results:** paste your metrics back to ArXivist -> Stage 6 Results Comparator

**Top implementation assumptions (from the SIR):**
- Training LR `6e-5` / cosine / 10% warmup / bf16 - *inferred, conf 0.62*
- Hyena implicit-filter MLP internals - *from Hyena reference, conf 0.55* (pretrained path avoids this)
- Mean-pooling + weight-tied head - *standard practice, conf 0.65*